# Question Generation Model Training

This notebook trains a T5 model for end-to-end question generation on the SQuAD dataset.

**Make sure to enable GPU:** Runtime → Change runtime type → GPU

## 1. Clone Repository

In [ ]:
!git clone https://github.com/saiteja2873/LLM.git
%cd LLM

## 2. Install Dependencies

In [ ]:
!pip install torch transformers datasets nltk accelerate sentencepiece tqdm

## 3. Verify GPU is Available

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 4. Prepare Training Data

This downloads SQuAD dataset and processes it for question generation.

In [ ]:
!python prepare_data.py --task e2e_qg --model_type t5

## 5. Train the Model

Training with T5-small for 3 epochs. Adjust parameters as needed:
- `--model_name_or_path`: Use `t5-base` for better quality (needs more memory)
- `--num_train_epochs`: More epochs = better quality but longer training
- `--per_device_train_batch_size`: Reduce if you get OOM errors

In [ ]:
!python run_qg.py \
    --model_name_or_path t5-small \
    --model_type t5 \
    --tokenizer_name_or_path t5_qg_tokenizer \
    --output_dir ./my_model \
    --train_file_path data/train_data_e2e_qg_highlight_t5.pt \
    --valid_file_path data/valid_data_e2e_qg_highlight_t5.pt \
    --per_device_train_batch_size 8 \
    --num_train_epochs 3 \
    --learning_rate 1e-4 \
    --save_steps 5000 \
    --logging_steps 500 \
    --do_train

## 6. Test the Trained Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load trained model
tokenizer = AutoTokenizer.from_pretrained("./my_model")
model = AutoModelForSeq2SeqLM.from_pretrained("./my_model")

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Test with sample text
text = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower. It was constructed from 1887 to 1889."

prompt = f"generate questions: {text}"
inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=4,
    early_stopping=True
)

questions = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Questions:")
print(questions)

## 7. Download Trained Model

Download the model to use locally.

In [ ]:
!zip -r my_model.zip my_model/

from google.colab import files
files.download('my_model.zip')

## 8. (Optional) Push to Hugging Face Hub

Share your model on Hugging Face.

In [ ]:
# Uncomment and run to push to Hugging Face
# from huggingface_hub import login
# login()  # Enter your HF token

# model.push_to_hub("your-username/question-generation-t5")
# tokenizer.push_to_hub("your-username/question-generation-t5")